In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
import os

import gfdl_utils.core as gu
import CM4Xutils
import cftime
import numpy as np
import pandas as pd
import xarray as xr
import xgcm
import xhistogram
import xwmt

import cmocean
import matplotlib.colors as colors
import matplotlib.ticker as mtick
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.dates as mdates
plt.rcParams.update({'font.size': 11})

### Load postprocessed model datasets

In [3]:
from common import *
grids = load_datasets()

No ideal age companion store for CM4Xp25 (missing ['CM4Xp25_historical_ideal_age_z.zarr', 'CM4Xp25_ssp585_ideal_age_z.zarr']); `agessc` will be absent from the CM4Xp25 dataset.
Inferring Z grid coordinate: depth `z_`


/work5/hfd/projects/CM4XTransientTracers/analysis/common.py:75: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  xr.open_zarr(f"../data/interim/{model}_historical_transient_tracers_surface.zarr"),


/work5/hfd/projects/CM4XTransientTracers/analysis/common.py:64: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  return xr.concat([xr.open_zarr(p) for p in paths], dim="year")


/work5/hfd/projects/CM4XTransientTracers/analysis/common.py:64: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  return xr.concat([xr.open_zarr(p) for p in paths], dim="year")


Inferring Z grid coordinate: depth `z_`


#### Compute year in which each grid cell switches from a CFC sink to source

In [4]:
sim = "CM4Xp125_forced"
ds = grids[sim]._ds
# One chunk along year: the 10-year rolling windows cannot span the
# one-year chunks that groupby leaves behind.
ds["fgcfc11_annual"] = ds["fgcfc11"].groupby("time.year").mean().chunk({"year": -1})
ds["fgcfc11_annual"] = ds["fgcfc11_annual"].where(ds["deptho"]!=0, np.nan)

always_sink = (ds["fgcfc11_annual"] > -1e-18).rolling({"year":10}, min_periods=1).mean("year").all("year").compute()
source_year = (ds["fgcfc11_annual"].rolling({"year":10}, min_periods=1).mean("year") <= -1e-18).idxmax("year")
source_year = source_year.where((~always_sink) & (ds.deptho!=0))
source_year = xr.where(source_year == 1850, 2100, source_year)

source_year.to_netcdf("../data/processed/cfc11_sink_to_source_transition_year.nc")